# Task 8B — Native physical-signal readiness

This notebook uses licensed data extracted into Colab-local storage (or optionally copies it from a personal Drive folder), creates a reviewable inventory, builds the Task 8B manifest and matched TIFF view, runs source/nuisance gates, validates PRNU without binary labels, and records a fail-closed retention decision. It does not download data, modify RINE, read the competition final test, or enable chromatic aberration automatically.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

PROJECT_ROOT = Path('/content/cya-techjam26')
SOURCE_MODE = 'local'  # 'local' or 'private_drive'
PRIVATE_DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/cya-techjam26-data')
LOCAL_DATA_ROOT = Path('/content/hackathon_data')
LOCAL_ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
DRIVE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
PRIVATE_DRIVE_TASK8B = PRIVATE_DRIVE_DATA_ROOT / 'raw/task8b'
LOCAL_TASK8B = LOCAL_DATA_ROOT / 'raw/task8b'

assert PROJECT_ROOT.is_dir(), f'Missing checkout: {PROJECT_ROOT}'
assert SOURCE_MODE in {'local', 'private_drive'}, 'Invalid SOURCE_MODE'

In [ ]:
if SOURCE_MODE == 'private_drive':
    assert PRIVATE_DRIVE_TASK8B.is_dir(), f'Missing personal Drive data: {PRIVATE_DRIVE_TASK8B}'
    LOCAL_TASK8B.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(PRIVATE_DRIVE_TASK8B, LOCAL_TASK8B, dirs_exist_ok=True)

assert (LOCAL_TASK8B / 'premier').is_dir(), f'Missing extracted PREMIER data: {LOCAL_TASK8B / "premier"}'
assert (LOCAL_TASK8B / 'genimage_ai').is_dir(), f'Missing extracted GenImage AI data: {LOCAL_TASK8B / "genimage_ai"}'
print('Using Task 8B source data at:', LOCAL_TASK8B)

In [ ]:
environment = os.environ.copy()
environment['DATA_ROOT'] = str(LOCAL_DATA_ROOT)
environment['ARTIFACT_ROOT'] = str(LOCAL_ARTIFACT_ROOT)
inventory = LOCAL_TASK8B / 'sources.csv'
if not inventory.is_file():
    subprocess.run(['make', 'task8b-inventory'], cwd=PROJECT_ROOT, env=environment, check=True)
    raise RuntimeError(
        'A draft sources.csv was created. Review it and inventory_preparation.json, '
        'correct any metadata, then rerun this cell.'
    )
subprocess.run(['make', 'task8b-prepare'], cwd=PROJECT_ROOT, env=environment, check=True)

In [ ]:
readiness_path = LOCAL_ARTIFACT_ROOT / 'task8b/audits/readiness_report.json'
readiness = json.loads(readiness_path.read_text(encoding='utf-8'))
print(json.dumps({key: readiness[key] for key in ('source_ready', 'training_ready', 'prnu_reference', 'chromatic_aberration')}, indent=2))
assert readiness['source_ready'], 'Source readiness failed; inspect readiness_report.json'
if readiness['prnu_reference']['ready']:
    subprocess.run(['make', 'task8b-prnu-references'], cwd=PROJECT_ROOT, env=environment, check=True)
else:
    print('PRNU references remain blocked by device/image coverage.')
subprocess.run(['make', 'task8b-matched'], cwd=PROJECT_ROOT, env=environment, check=True)
subprocess.run(['make', 'task8b-prnu-validate'], cwd=PROJECT_ROOT, env=environment, check=True)
subprocess.run(['make', 'task8b-decision'], cwd=PROJECT_ROOT, env=environment, check=True)
decision_path = LOCAL_ARTIFACT_ROOT / 'task8b/reports/retention_decision.json'
decision = json.loads(decision_path.read_text(encoding='utf-8'))
print(json.dumps(decision, indent=2))
if not decision['fusion_training_eligible']:
    print('Task 8B is complete with no fusion run; no physical estimator passed its independent gate.')

In [ ]:
local_results = LOCAL_ARTIFACT_ROOT / 'task8b'
drive_results = DRIVE_ARTIFACT_ROOT / 'task8b'
if Path('/content/drive/MyDrive').is_dir():
    drive_results.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_results, drive_results, dirs_exist_ok=True)
    print('Synced Task 8B artifacts to:', drive_results)
else:
    print('Drive is not mounted; artifacts remain at:', local_results)